## EXTRACTION OF CLUB INFORMATION

In [15]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
import time
import random
from urllib.parse import quote
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
import time
import random
import re

In [9]:
# ========= CONFIGURATION =========
INPUT_CSV = "clubs.csv"  
OUTPUT_CSV = "clubs_with_urls.csv"  # Output with URLs added

In [3]:
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/141.0.0.0 Safari/537.36"
}


In [6]:
##FIRST CONSTRUCT URLs

In [11]:
"""
 BUILD TRANSFERMARKT URLS FOR CLUBS
This part searches for club IDs and builds the Transfermarkt URLs
"""

def clean_club_name_for_url(club_name):
    """Clean club name for URL slug"""
    # Fix common encoding issues
    name = club_name.replace("Ã­", "i").replace("Ã¡", "a").replace("Ã©", "e")
    name = name.replace("Ã³", "o").replace("Ãº", "u").replace("Ã±", "n")
    name = name.replace("Ã¼", "u").replace("Ã§", "c")
    
    # Convert to lowercase
    name = name.lower().strip()
    
    # Replace spaces with hyphens
    name = name.replace(' ', '-')
    
    # Remove special characters, keep only alphanumeric and hyphens
    name = re.sub(r'[^a-z0-9-]', '', name)
    
    # Remove multiple hyphens
    name = re.sub(r'-+', '-', name)
    
    return name

def search_club_id_and_url(club_name):
    """Search for club on Transfermarkt and return club ID and URL"""
    print(f"   🔍 Searching: {club_name}")
    
    try:
        # Clean name for search
        search_name = clean_club_name_for_url(club_name).replace('-', ' ')
        search_url = f"https://www.transfermarkt.com/schnellsuche/ergebnis/schnellsuche?query={quote(search_name)}"
        
        response = requests.get(search_url, headers=HEADERS, timeout=15)
        if response.status_code != 200:
            return None, None
        
        soup = BeautifulSoup(response.content, "html.parser")
        
        # Method 1: Look for result-table
        result_table = soup.find('table', class_='result-table')
        if result_table:
            rows = result_table.find_all('tr')
            
            for row in rows:
                # Look for club links
                links = row.find_all('a', href=re.compile(r'/verein/\d+'))
                for link in links:
                    href = link.get('href', '')
                    club_id_match = re.search(r'/verein/(\d+)', href)
                    if club_id_match:
                        club_id = club_id_match.group(1)
                        link_text = link.get_text(strip=True).lower()
                        search_name_lower = search_name.lower()
                        
                        # Check if this is likely the right club
                        if (search_name_lower in link_text or 
                            link_text in search_name_lower or
                            any(word in link_text for word in search_name_lower.split()[:2])):
                            
                            club_url = f"https://www.transfermarkt.com{href}"
                            print(f"   ✅ Found ID: {club_id}")
                            return club_id, club_url
        
        # Method 2: Look for any verein link in the page
        all_verein_links = soup.find_all('a', href=re.compile(r'/verein/\d+'))
        for link in all_verein_links[:5]:
            href = link.get('href', '')
            club_id_match = re.search(r'/verein/(\d+)', href)
            if club_id_match:
                club_id = club_id_match.group(1)
                link_text = link.get_text(strip=True).lower()
                search_name_lower = search_name.lower()
                
                if (search_name_lower in link_text or 
                    search_name_lower.split()[0] in link_text):
                    club_url = f"https://www.transfermarkt.com{href}"
                    print(f"   ✅ Found ID: {club_id}")
                    return club_id, club_url
        
        print(f"   ❌ No ID found")
        return None, None
        
    except Exception as e:
        print(f"   ❌ Error: {str(e)[:80]}")
        return None, None

def build_transfermarkt_url_from_id(club_name, club_id):
    """Build Transfermarkt URL from club ID"""
    if not club_id:
        return 'Not found'
    
    slug = clean_club_name_for_url(club_name)
    url = f"https://www.transfermarkt.com/{slug}/startseite/verein/{club_id}"
    return url

def add_urls_to_clubs():
    """Add Transfermarkt URLs to clubs CSV"""
    print("🚀 BUILDING TRANSFERMARKT URLS FOR CLUBS")
    print("=" * 70)
    
    try:
        # Try different encodings
        encodings = ['utf-8', 'latin-1', 'iso-8859-1', 'cp1252', 'windows-1252']
        df = None
        
        for encoding in encodings:
            try:
                df = pd.read_csv(INPUT_CSV, encoding=encoding)
                print(f"✅ Loaded CSV with {encoding} encoding")
                break
            except UnicodeDecodeError:
                continue
        
        if df is None:
            df = pd.read_csv(INPUT_CSV, encoding='utf-8', errors='replace')
            print("⚠️ Loaded CSV with error replacement")
        
        # Find the club name column
        name_column = None
        possible_name_columns = ['squad', 'club_name', 'Club', 'name', 'Name', 'club', 'team', 'Team']
        
        for col in possible_name_columns:
            if col in df.columns:
                name_column = col
                break
        
        if not name_column:
            print(f"📋 Available columns: {list(df.columns)}")
            name_column = df.columns[0]
            print(f"⚠️ Using first column as club name: '{name_column}'")
        
        print(f"📁 Loaded {len(df)} clubs")
        print(f"📊 Using column: '{name_column}' for club names")
        
        # Add URL columns
        if 'transfermarkt_id' not in df.columns:
            df['transfermarkt_id'] = 'Not found'
        if 'transfermarkt_url' not in df.columns:
            df['transfermarkt_url'] = 'Not found'
        
        found_count = 0
        failed_clubs = []
        
        for index, row in df.iterrows():
            club_name = str(row[name_column]).strip()
            
            print(f"\n[{index + 1}/{len(df)}] {club_name}")
            
            # Skip if already has URL
            if str(df.at[index, 'transfermarkt_url']).startswith('https://www.transfermarkt.com'):
                print(f"   ⏭️ Already has URL, skipping")
                found_count += 1
                continue
            
            # Skip invalid names
            if not club_name or club_name == 'nan' or len(club_name) < 2:
                print(f"   ⏭️ Invalid club name, skipping")
                continue
            
            # Search for club ID and URL
            club_id, club_url = search_club_id_and_url(club_name)
            
            if club_id and club_url:
                df.at[index, 'transfermarkt_id'] = club_id
                df.at[index, 'transfermarkt_url'] = club_url
                found_count += 1
                print(f"   ✅ URL: {club_url}")
            else:
                failed_clubs.append(club_name)
                print(f"   ❌ Could not find URL")
            
            # Save progress every 20 clubs
            if (index + 1) % 20 == 0:
                progress_file = f"urls_progress_{index + 1}.csv"
                df.to_csv(progress_file, index=False)
                print(f"\n💾 Progress saved: {index + 1} clubs processed")
                print(f"📊 Found so far: {found_count}/{index + 1} ({found_count/(index + 1)*100:.1f}%)")
            
            # Delay between requests
            if index + 1 < len(df):
                delay = random.uniform(2, 4)
                print(f"⏳ Waiting {delay:.1f}s...")
                time.sleep(delay)
        
        # Save final results
        df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')
        
        # Summary
        print("\n" + "=" * 70)
        print("🎉 URL BUILDING COMPLETED!")
        print("=" * 70)
        print(f"📊 Total clubs: {len(df)}")
        print(f"✅ URLs found: {found_count}")
        print(f"❌ URLs not found: {len(df) - found_count}")
        print(f"📈 Success rate: {found_count/len(df)*100:.1f}%")
        
        if failed_clubs:
            print(f"\n⚠️ Clubs not found (may need manual lookup):")
            for club in failed_clubs[:30]:
                print(f"   - {club}")
            if len(failed_clubs) > 30:
                print(f"   ... and {len(failed_clubs) - 30} more")
            print(f"\n💡 Tip: Save these to a separate file for manual ID lookup")
            
            # Save failed clubs to a separate file
            failed_df = pd.DataFrame({'club_name': failed_clubs})
            failed_df.to_csv("clubs_not_found.csv", index=False)
            print(f"💾 Failed clubs saved to: clubs_not_found.csv")
        
        print(f"\n💾 Results saved to: {OUTPUT_CSV}")
        
        # Show sample of created URLs
        sample_df = df[df['transfermarkt_url'] != 'Not found'].head(10)
        if len(sample_df) > 0:
            print(f"\n🎯 SAMPLE CREATED URLS:")
            for _, club in sample_df.iterrows():
                print(f"   • {club[name_column]} → {club['transfermarkt_url']}")
        
        return df
        
    except FileNotFoundError:
        print(f"❌ File '{INPUT_CSV}' not found")
        print("   Please make sure the CSV file is in the same directory")
        return None
    except Exception as e:
        print(f"❌ Error: {e}")
        return None

def check_csv_structure():
    """Quick function to check your CSV structure"""
    print("🔍 CHECKING CSV STRUCTURE")
    print("=" * 60)
    
    try:
        df = pd.read_csv(INPUT_CSV)
        print(f"✅ File '{INPUT_CSV}' loaded successfully")
        print(f"📊 Total rows: {len(df)}")
        print(f"📋 Columns: {list(df.columns)}")
        print(f"\n👀 First 5 rows:")
        print(df.head(5))
        
        # Suggest which column might be club names
        print(f"\n🎯 Suggested column for club names:")
        for col in df.columns:
            if any(word in col.lower() for word in ['squad', 'club', 'team', 'name']):
                print(f"   - '{col}' (likely club names)")
        
        return df
    except FileNotFoundError:
        print(f"❌ File '{INPUT_CSV}' not found")
        return None
    except Exception as e:
        print(f"❌ Error: {e}")
        return None

# ========= RUN THE CODE =========
if __name__ == "__main__":
    print("Choose an option:")
    print("1. Check CSV structure first")
    print("2. Build URLs directly")
    
    choice = input("Enter 1 or 2: ").strip()
    
    if choice == "1":
        check_csv_structure()
        print("\n" + "=" * 60)
        proceed = input("Proceed with URL building? (y/n): ").strip().lower()
        if proceed == 'y':
            add_urls_to_clubs()
    else:
        add_urls_to_clubs()

Choose an option:
1. Check CSV structure first
2. Build URLs directly


🚀 BUILDING TRANSFERMARKT URLS FOR CLUBS
✅ Loaded CSV with utf-8 encoding
📁 Loaded 1346 clubs
📊 Using column: 'squad' for club names

[1/1346] Real Madrid B
   🔍 Searching: Real Madrid B
   ❌ No ID found
   ❌ Could not find URL
⏳ Waiting 2.6s...

[2/1346] AlmerÃ­a
   🔍 Searching: AlmerÃ­a
   ❌ No ID found
   ❌ Could not find URL
⏳ Waiting 2.2s...

[3/1346] Sevilla
   🔍 Searching: Sevilla
   ❌ No ID found
   ❌ Could not find URL
⏳ Waiting 3.6s...

[4/1346] Manchester City
   🔍 Searching: Manchester City
   ✅ Found ID: 281
   ✅ URL: https://www.transfermarkt.com/manchester-city/startseite/verein/281
⏳ Waiting 2.4s...

[5/1346] Valencia
   🔍 Searching: Valencia
   ❌ No ID found
   ❌ Could not find URL
⏳ Waiting 3.7s...

[6/1346] Middlesbrough
   🔍 Searching: Middlesbrough
   ✅ Found ID: 641
   ✅ URL: https://www.transfermarkt.com/fc-middlesbrough/startseite/verein/641
⏳ Waiting 2.3s...

[7/1346] BeÅŸiktaÅŸ
   🔍 Searching: BeÅŸiktaÅŸ
   ❌ No ID found
   ❌ Could not find URL
⏳ Waiting 2.9s..

In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
import time
import random
from urllib.parse import quote, unquote
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException

# ========= CONFIGURATION =========
INPUT_CSV = "clubs_missing.csv"  
OUTPUT_CSV = "clubs_with_urls2.csv"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/141.0.0.0 Safari/537.36"
}

# Manual mapping for clubs that are hard to find
MANUAL_MAPPING = {
    'Real Madrid B': 'Real Madrid Castilla',
    'Almería': 'Almeria',
    'Sevilla': 'Sevilla FC',
    'Manchester City': 'Manchester City',
}

# Known club IDs for popular teams (backup)
KNOWN_CLUB_IDS = {
    'Real Madrid B': 124,  # Real Madrid Castilla
    'Almería': 108,  # UD Almería
    'Sevilla': 184,  # Sevilla FC
    'Manchester City': 281,
}

def clean_club_name_for_url(club_name):
    """Clean club name for URL slug"""
    # Fix common encoding issues
    name = club_name.replace("Ã­", "i").replace("Ã¡", "a").replace("Ã©", "e")
    name = name.replace("Ã³", "o").replace("Ãº", "u").replace("Ã±", "n")
    name = name.replace("Ã¼", "u").replace("Ã§", "c").replace("Ã", "")
    
    # Remove accents and special characters
    import unicodedata
    name = unicodedata.normalize('NFKD', name).encode('ASCII', 'ignore').decode('ASCII')
    
    # Convert to lowercase
    name = name.lower().strip()
    
    # Replace spaces with hyphens
    name = name.replace(' ', '-')
    
    # Remove special characters, keep only alphanumeric and hyphens
    name = re.sub(r'[^a-z0-9-]', '', name)
    
    # Remove multiple hyphens
    name = re.sub(r'-+', '-', name)
    
    # Remove trailing/leading hyphens
    name = name.strip('-')
    
    return name

def search_by_google_site_search(club_name):
    """Use Google site search to find Transfermarkt URL"""
    try:
        search_query = f"site:transfermarkt.com {club_name} verein"
        google_url = f"https://www.google.com/search?q={quote(search_query)}"
        
        response = requests.get(google_url, headers=HEADERS, timeout=10)
        if response.status_code != 200:
            return None, None
        
        soup = BeautifulSoup(response.content, "html.parser")
        
        # Look for Transfermarkt links in Google results
        for link in soup.find_all('a', href=True):
            href = link.get('href', '')
            if 'transfermarkt.com' in href and '/verein/' in href:
                # Clean Google redirect URL
                match = re.search(r'(https?://[^&]+)', href)
                if match:
                    url = match.group(1)
                    club_id_match = re.search(r'/verein/(\d+)', url)
                    if club_id_match:
                        return club_id_match.group(1), url
        
        return None, None
    except:
        return None, None

def search_by_direct_url_pattern(club_name):
    """Try direct URL patterns for common clubs"""
    # Common team name variations
    name_variations = [
        club_name,
        club_name.replace('B', 'Castilla'),
        club_name.replace('FC', '').strip(),
        f"{club_name} FC",
        club_name.replace(' ', '-').lower()
    ]
    
    for variation in name_variations[:3]:
        slug = clean_club_name_for_url(variation)
        
        # Try common Transfermarkt URL patterns
        url_patterns = [
            f"https://www.transfermarkt.com/{slug}/startseite/verein/",
            f"https://www.transfermarkt.com/{slug}/startseite/verein",
            f"https://www.transfermarkt.com/{slug}/verein/",
        ]
        
        for url_pattern in url_patterns:
            try:
                response = requests.get(url_pattern, headers=HEADERS, timeout=5)
                if response.status_code == 200:
                    # Check if we got a valid club page
                    soup = BeautifulSoup(response.content, "html.parser")
                    
                    # Look for club ID in the page
                    verein_match = re.search(r'/verein/(\d+)', response.url)
                    if verein_match:
                        club_id = verein_match.group(1)
                        return club_id, response.url
            except:
                continue
    
    return None, None

def search_by_selenium(club_name):
    """Use Selenium for JavaScript-heavy pages"""
    chrome_options = Options()
    chrome_options.add_argument("--headless")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    
    driver = None
    try:
        driver = webdriver.Chrome(options=chrome_options)
        search_url = f"https://www.transfermarkt.com/schnellsuche/ergebnis/schnellsuche?query={quote(club_name)}"
        driver.get(search_url)
        
        # Wait for results
        wait = WebDriverWait(driver, 10)
        wait.until(EC.presence_of_element_located((By.TAG_NAME, "body")))
        
        time.sleep(2)
        
        # Find links to club pages
        links = driver.find_elements(By.CSS_SELECTOR, "a[href*='/verein/']")
        
        for link in links[:5]:
            href = link.get_attribute('href')
            link_text = link.text.lower()
            club_name_lower = club_name.lower()
            
            if (club_name_lower in link_text or 
                link_text in club_name_lower or
                any(word in link_text for word in club_name_lower.split()[:2])):
                
                club_id_match = re.search(r'/verein/(\d+)', href)
                if club_id_match:
                    return club_id_match.group(1), href
        
        return None, None
        
    except Exception as e:
        print(f"   Selenium error: {str(e)[:80]}")
        return None, None
    finally:
        if driver:
            driver.quit()

def search_club_id_and_url(club_name, use_selenium=False):
    """Search for club on Transfermarkt using multiple methods"""
    print(f"   🔍 Searching: {club_name}")
    
    # Method 0: Check manual mapping first
    if club_name in MANUAL_MAPPING:
        club_name = MANUAL_MAPPING[club_name]
        print(f"   📝 Using mapped name: {club_name}")
    
    # Method 0.1: Check known IDs
    if club_name in KNOWN_CLUB_IDS:
        club_id = KNOWN_CLUB_IDS[club_name]
        slug = clean_club_name_for_url(club_name)
        club_url = f"https://www.transfermarkt.com/{slug}/startseite/verein/{club_id}"
        print(f"   ✅ Using known ID: {club_id}")
        return club_id, club_url
    
    try:
        # Method 1: Standard search
        search_name = clean_club_name_for_url(club_name).replace('-', ' ')
        search_url = f"https://www.transfermarkt.com/schnellsuche/ergebnis/schnellsuche?query={quote(search_name)}"
        
        response = requests.get(search_url, headers=HEADERS, timeout=15)
        if response.status_code == 200:
            soup = BeautifulSoup(response.content, "html.parser")
            
            # Look for result-table
            result_table = soup.find('table', class_='result-table')
            if result_table:
                rows = result_table.find_all('tr')
                
                for row in rows:
                    links = row.find_all('a', href=re.compile(r'/verein/\d+'))
                    for link in links:
                        href = link.get('href', '')
                        club_id_match = re.search(r'/verein/(\d+)', href)
                        if club_id_match:
                            club_id = club_id_match.group(1)
                            link_text = link.get_text(strip=True).lower()
                            search_name_lower = search_name.lower()
                            
                            if (search_name_lower in link_text or 
                                link_text in search_name_lower or
                                any(word in link_text for word in search_name_lower.split()[:2])):
                                
                                club_url = f"https://www.transfermarkt.com{href}"
                                print(f"   ✅ Found ID: {club_id} (Method 1)")
                                return club_id, club_url
        
        # Method 2: Direct URL pattern
        club_id, club_url = search_by_direct_url_pattern(club_name)
        if club_id:
            print(f"   ✅ Found ID: {club_id} (Method 2)")
            return club_id, club_url
        
        # Method 3: Google site search
        club_id, club_url = search_by_google_site_search(club_name)
        if club_id:
            print(f"   ✅ Found ID: {club_id} (Method 3)")
            return club_id, club_url
        
        # Method 4: Try with 'FC' suffix
        if not club_name.endswith('FC') and not club_name.endswith('B'):
            club_id, club_url = search_by_google_site_search(f"{club_name} FC")
            if club_id:
                print(f"   ✅ Found ID: {club_id} (Method 4 - with FC)")
                return club_id, club_url
        
        # Method 5: Selenium (optional, slower)
        if use_selenium:
            club_id, club_url = search_by_selenium(club_name)
            if club_id:
                print(f"   ✅ Found ID: {club_id} (Method 5 - Selenium)")
                return club_id, club_url
        
        print(f"   ❌ No ID found after all methods")
        return None, None
        
    except Exception as e:
        print(f"   ❌ Error: {str(e)[:80]}")
        return None, None

def build_transfermarkt_url_from_id(club_name, club_id):
    """Build Transfermarkt URL from club ID"""
    if not club_id or club_id == 'Not found':
        return 'Not found'
    
    # For B teams, use the main team name
    if club_name.endswith('B'):
        main_name = club_name.replace(' B', ' Castilla').replace('B', 'Castilla')
        slug = clean_club_name_for_url(main_name)
    else:
        slug = clean_club_name_for_url(club_name)
    
    url = f"https://www.transfermarkt.com/{slug}/startseite/verein/{club_id}"
    return url

def add_urls_to_clubs():
    """Add Transfermarkt URLs to clubs CSV"""
    print("🚀 ENHANCED URL FINDER FOR CLUBS")
    print("=" * 70)
    
    try:
        # Try different encodings
        encodings = ['utf-8', 'latin-1', 'iso-8859-1', 'cp1252', 'windows-1252']
        df = None
        
        for encoding in encodings:
            try:
                df = pd.read_csv(INPUT_CSV, encoding=encoding)
                print(f"✅ Loaded CSV with {encoding} encoding")
                break
            except UnicodeDecodeError:
                continue
        
        if df is None:
            df = pd.read_csv(INPUT_CSV, encoding='utf-8', errors='replace')
            print("⚠️ Loaded CSV with error replacement")
        
        # Find the club name column
        name_column = None
        possible_name_columns = ['squad', 'club_name', 'Club', 'name', 'Name', 'club', 'team', 'Team']
        
        for col in possible_name_columns:
            if col in df.columns:
                name_column = col
                break
        
        if not name_column:
            print(f"📋 Available columns: {list(df.columns)}")
            name_column = df.columns[0]
            print(f"⚠️ Using first column as club name: '{name_column}'")
        
        print(f"📁 Loaded {len(df)} clubs")
        print(f"📊 Using column: '{name_column}' for club names")
        
        # Add URL columns
        if 'transfermarkt_id' not in df.columns:
            df['transfermarkt_id'] = 'Not found'
        if 'transfermarkt_url' not in df.columns:
            df['transfermarkt_url'] = 'Not found'
        
        found_count = 0
        failed_clubs = []
        
        # Ask if user wants to use Selenium (slower but more accurate)
        use_selenium = input("\nUse Selenium for better search? (y/n, slower but more accurate): ").strip().lower() == 'y'
        
        for index, row in df.iterrows():
            club_name = str(row[name_column]).strip()
            
            print(f"\n[{index + 1}/{len(df)}] {club_name}")
            
            # Skip if already has URL
            if str(df.at[index, 'transfermarkt_url']).startswith('https://www.transfermarkt.com'):
                print(f"   ⏭️ Already has URL, skipping")
                found_count += 1
                continue
            
            # Skip invalid names
            if not club_name or club_name == 'nan' or len(club_name) < 2:
                print(f"   ⏭️ Invalid club name, skipping")
                continue
            
            # Search for club ID and URL
            club_id, club_url = search_club_id_and_url(club_name, use_selenium=use_selenium)
            
            if club_id and club_url:
                df.at[index, 'transfermarkt_id'] = club_id
                df.at[index, 'transfermarkt_url'] = club_url
                found_count += 1
                print(f"   ✅ URL: {club_url}")
            else:
                failed_clubs.append(club_name)
                print(f"   ❌ Could not find URL")
            
            # Save progress every 10 clubs
            if (index + 1) % 10 == 0:
                progress_file = f"urls_progress_{index + 1}.csv"
                df.to_csv(progress_file, index=False)
                print(f"\n💾 Progress saved: {index + 1} clubs processed")
                print(f"📊 Found so far: {found_count}/{index + 1} ({found_count/(index + 1)*100:.1f}%)")
            
            # Delay between requests (shorter if using known IDs)
            if index + 1 < len(df) and not (club_name in KNOWN_CLUB_IDS):
                delay = random.uniform(1.5, 3)
                print(f"⏳ Waiting {delay:.1f}s...")
                time.sleep(delay)
        
        # Save final results
        df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')
        
        # Summary
        print("\n" + "=" * 70)
        print("🎉 URL FINDING COMPLETED!")
        print("=" * 70)
        print(f"📊 Total clubs: {len(df)}")
        print(f"✅ URLs found: {found_count}")
        print(f"❌ URLs not found: {len(df) - found_count}")
        print(f"📈 Success rate: {found_count/len(df)*100:.1f}%")
        
        if failed_clubs:
            print(f"\n⚠️ Clubs still not found (need manual lookup):")
            for club in failed_clubs[:30]:
                print(f"   - {club}")
            if len(failed_clubs) > 30:
                print(f"   ... and {len(failed_clubs) - 30} more")
            
            # Save failed clubs to a separate file
            failed_df = pd.DataFrame({'club_name': failed_clubs})
            failed_df.to_csv("clubs_still_not_found.csv", index=False)
            print(f"\n💾 Failed clubs saved to: clubs_still_not_found.csv")
            print(f"\n💡 Tips for manual lookup:")
            print(f"   1. Visit: https://www.transfermarkt.com/schnellsuche/ergebnis/schnellsuche?query=clubname")
            print(f"   2. Search for the club manually")
            print(f"   3. Get the ID from URL: /verein/XXXXX")
            print(f"   4. Add to KNOWN_CLUB_IDS dictionary in the code")
        
        print(f"\n💾 Results saved to: {OUTPUT_CSV}")
        
        # Show sample of created URLs
        sample_df = df[df['transfermarkt_url'] != 'Not found'].head(10)
        if len(sample_df) > 0:
            print(f"\n🎯 SAMPLE CREATED URLS:")
            for _, club in sample_df.iterrows():
                print(f"   • {club[name_column]} → {club['transfermarkt_url']}")
        
        return df
        
    except FileNotFoundError:
        print(f"❌ File '{INPUT_CSV}' not found")
        print("   Please make sure the CSV file is in the same directory")
        return None
    except Exception as e:
        print(f"❌ Error: {e}")
        return None

# ========= RUN THE CODE =========
if __name__ == "__main__":
    print("🔍 ENHANCED TRANSFERMARKT URL FINDER")
    print("This version uses multiple search methods\n")
    
    add_urls_to_clubs()


🔍 ENHANCED TRANSFERMARKT URL FINDER
This version uses multiple search methods

🚀 ENHANCED URL FINDER FOR CLUBS
✅ Loaded CSV with utf-8 encoding
📁 Loaded 499 clubs
📊 Using column: 'squad' for club names

[1/499] Real Madrid B
   🔍 Searching: Real Madrid B
   📝 Using mapped name: Real Madrid Castilla
   ✅ Found ID: 6767 (Method 5 - Selenium)
   ✅ URL: https://www.transfermarkt.com/real-madrid-b-castilla-/startseite/verein/6767

[2/499] AlmerÃ­a
   🔍 Searching: AlmerÃ­a
   ❌ No ID found after all methods
   ❌ Could not find URL
⏳ Waiting 1.9s...

[3/499] Sevilla
   🔍 Searching: Sevilla
   📝 Using mapped name: Sevilla FC
   ✅ Found ID: 368 (Method 5 - Selenium)
   ✅ URL: https://www.transfermarkt.com/fc-sevilla/startseite/verein/368

[4/499] Valencia
   🔍 Searching: Valencia
   ✅ Found ID: 515 (Method 5 - Selenium)
   ✅ URL: https://www.transfermarkt.com/vereinslos/startseite/verein/515
⏳ Waiting 2.8s...

[5/499] BeÅŸiktaÅŸ
   🔍 Searching: BeÅŸiktaÅŸ
   ❌ No ID found after all methods
   ❌

KeyboardInterrupt: 

In [2]:
"""
TRANSFERMARKT URL FINDER - IMPROVED VERSION
Handles "Not found" clubs with better fuzzy matching and multiple fallback strategies.

Usage:
    python find_transfermarkt_urls.py

Input:  clubs_with_urls.csv  (your existing file with some "Not found" rows)
Output: clubs_final.csv      (same file with new URLs filled in where possible)
        clubs_still_missing.csv  (clubs that still couldn't be found)
"""

import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
import time
import random
from urllib.parse import quote
import unicodedata

# ── CONFIG ──────────────────────────────────────────────────────────────────
INPUT_CSV  = "clubs_missing.csv"   # your current file (may have "Not found")
OUTPUT_CSV = "clubs_final.csv"
MISSING_CSV = "clubs_still_missing.csv"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}

DELAY_RANGE = (2.5, 5.0)   # seconds between requests – be polite
SAVE_EVERY  = 20           # save progress every N clubs
# ────────────────────────────────────────────────────────────────────────────


# ── TEXT HELPERS ─────────────────────────────────────────────────────────────

def fix_encoding(text: str) -> str:
    """Fix common mojibake patterns that appear when latin-1 text is read as utf-8."""
    replacements = {
        "Ã­": "í", "Ã¡": "á", "Ã©": "é", "Ã³": "ó", "Ãº": "ú",
        "Ã±": "ñ", "Ã¼": "ü", "Ã§": "ç", "Ã ": "à", "Ã¨": "è",
        "Ã¦": "æ", "Ã¸": "ø", "Ã¥": "å", "Ã„": "Ä", "Ã–": "Ö",
        "Ãœ": "Ü", "ÃŸ": "ß", "Ã‚": "Â", "Ã‡": "Ç", "Ã‰": "É",
        "Ã€": "À", "Ã‹": "Ë", "Ã": "Í", "Ã™": "Ù", "Ã›": "Û",
    }
    for bad, good in replacements.items():
        text = text.replace(bad, good)
    return text


def to_ascii(text: str) -> str:
    """Transliterate accented characters to plain ASCII (for slug / search)."""
    return unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("ascii")


def make_slug(name: str) -> str:
    """Create the URL slug used by Transfermarkt."""
    name = fix_encoding(name)
    name = to_ascii(name).lower().strip()
    name = name.replace(" ", "-")
    name = re.sub(r"[^a-z0-9-]", "", name)
    name = re.sub(r"-+", "-", name).strip("-")
    return name


def name_tokens(name: str) -> set:
    """Return a set of meaningful lowercase tokens from a club name."""
    name = fix_encoding(name)
    name = to_ascii(name).lower()
    # Remove common noise words
    stop = {"fc", "cf", "sc", "ac", "as", "us", "sv", "bv", "sk", "fk",
            "the", "de", "del", "la", "el", "los", "las", "1.", "2."}
    tokens = set(re.findall(r"[a-z0-9]+", name)) - stop
    return tokens


def token_overlap(a: str, b: str) -> float:
    """Jaccard-like token overlap between two club name strings."""
    ta, tb = name_tokens(a), name_tokens(b)
    if not ta or not tb:
        return 0.0
    return len(ta & tb) / len(ta | tb)


# ── SEARCH STRATEGIES ────────────────────────────────────────────────────────

def _extract_verein_links(soup: BeautifulSoup):
    """Return list of (club_id, link_text, href) from any soup."""
    results = []
    for tag in soup.find_all("a", href=re.compile(r"/verein/\d+")):
        href = tag.get("href", "")
        m = re.search(r"/verein/(\d+)", href)
        if m:
            results.append((m.group(1), tag.get_text(strip=True), href))
    return results


def _best_match(links, club_name: str, threshold: float = 0.25):
    """Pick the best matching (club_id, href) from a list of (id, text, href) tuples."""
    best_score = -1
    best = (None, None)
    for club_id, link_text, href in links:
        score = token_overlap(club_name, link_text)
        if score > best_score:
            best_score = score
            best = (club_id, href)
    if best_score >= threshold:
        return best
    return (None, None)


def search_schnellsuche(club_name: str):
    """Strategy 1 – Transfermarkt quick-search endpoint."""
    query = to_ascii(fix_encoding(club_name))
    url = (
        "https://www.transfermarkt.com/schnellsuche/ergebnis/schnellsuche"
        f"?query={quote(query)}"
    )
    try:
        r = requests.get(url, headers=HEADERS, timeout=15)
        if r.status_code != 200:
            return None, None
        soup = BeautifulSoup(r.content, "html.parser")
        links = _extract_verein_links(soup)
        return _best_match(links, club_name, threshold=0.20)
    except Exception as e:
        print(f"      [schnellsuche error] {e}")
        return None, None


def search_schnellsuche_short(club_name: str):
    """Strategy 2 – Use only the first two meaningful tokens as query (helps with long names)."""
    tokens = list(name_tokens(club_name))[:2]
    if not tokens:
        return None, None
    query = " ".join(tokens)
    url = (
        "https://www.transfermarkt.com/schnellsuche/ergebnis/schnellsuche"
        f"?query={quote(query)}"
    )
    try:
        r = requests.get(url, headers=HEADERS, timeout=15)
        if r.status_code != 200:
            return None, None
        soup = BeautifulSoup(r.content, "html.parser")
        links = _extract_verein_links(soup)
        return _best_match(links, club_name, threshold=0.20)
    except Exception as e:
        print(f"      [schnellsuche_short error] {e}")
        return None, None


def search_google_style(club_name: str):
    """Strategy 3 – Try fetching a guessed slug URL directly on Transfermarkt."""
    slug = make_slug(club_name)
    # Transfermarkt startseite pages redirect correctly even with a wrong ID
    # We try a few known ID ranges by guessing nothing – instead we use their
    # search with site: hint embedded in the query (no external Google needed).
    url = (
        "https://www.transfermarkt.com/schnellsuche/ergebnis/schnellsuche"
        f"?query={quote(slug.replace('-', ' '))}&Kat=Verein"
    )
    try:
        r = requests.get(url, headers=HEADERS, timeout=15)
        if r.status_code != 200:
            return None, None
        soup = BeautifulSoup(r.content, "html.parser")
        links = _extract_verein_links(soup)
        return _best_match(links, club_name, threshold=0.15)
    except Exception as e:
        print(f"      [google_style error] {e}")
        return None, None


def search_reserve_team(club_name: str):
    """Strategy 4 – For reserve / B teams, search the parent club name."""
    # Strip common suffixes: B, II, Reserve, U23, U21, etc.
    stripped = re.sub(
        r"\b(b|ii|iii|u\d{2}|reserve|reserves|youth|sub)\b", "",
        club_name, flags=re.IGNORECASE
    ).strip(" -")
    if stripped.lower() == club_name.lower():
        return None, None   # nothing changed, skip
    print(f"      [reserve] trying parent name: '{stripped}'")
    return search_schnellsuche(stripped)


# ── MAIN SEARCH ORCHESTRATOR ─────────────────────────────────────────────────

def find_club(club_name: str):
    """
    Run all strategies in order. Return (club_id, full_url) or (None, None).
    """
    strategies = [
        ("full name",    lambda: search_schnellsuche(club_name)),
        ("short tokens", lambda: search_schnellsuche_short(club_name)),
        ("slug query",   lambda: search_google_style(club_name)),
        ("parent club",  lambda: search_reserve_team(club_name)),
    ]

    for label, fn in strategies:
        print(f"   → [{label}]", end=" ", flush=True)
        club_id, href = fn()
        if club_id and href:
            # Build canonical URL
            slug = make_slug(club_name)
            full_url = f"https://www.transfermarkt.com{href}"
            print(f"found! ID={club_id}")
            return club_id, full_url
        print("no match")
        time.sleep(random.uniform(0.8, 1.5))   # small pause between strategies

    return None, None


# ── CSV PROCESSING ────────────────────────────────────────────────────────────

def load_csv(path: str) -> pd.DataFrame:
    for enc in ["utf-8", "latin-1", "cp1252", "iso-8859-1"]:
        try:
            df = pd.read_csv(path, encoding=enc)
            print(f"  Loaded '{path}' ({enc}), {len(df)} rows")
            return df
        except (UnicodeDecodeError, FileNotFoundError):
            continue
    df = pd.read_csv(path, encoding="utf-8", errors="replace")
    print(f"  Loaded '{path}' (utf-8 with replacements), {len(df)} rows")
    return df


def get_name_column(df: pd.DataFrame) -> str:
    for col in ["squad", "club_name", "Club", "name", "Name", "club", "team", "Team"]:
        if col in df.columns:
            return col
    return df.columns[0]


def run():
    print("=" * 70)
    print("  TRANSFERMARKT URL FINDER – IMPROVED")
    print("=" * 70)

    df = load_csv(INPUT_CSV)
    name_col = get_name_column(df)
    print(f"  Name column: '{name_col}'")

    # Ensure URL columns exist
    for col in ["transfermarkt_id", "transfermarkt_url"]:
        if col not in df.columns:
            df[col] = "Not found"

    # Identify rows that still need work
    needs_url = df["transfermarkt_url"].apply(
        lambda v: not str(v).startswith("https://")
    )
    todo_indices = df[needs_url].index.tolist()
    print(f"  Clubs to process: {len(todo_indices)} / {len(df)}")
    print()

    found_count  = 0
    failed_clubs = []

    for i, idx in enumerate(todo_indices, 1):
        club_name = fix_encoding(str(df.at[idx, name_col]).strip())
        print(f"[{i}/{len(todo_indices)}] {club_name}")

        if not club_name or club_name in ("nan", "") or len(club_name) < 2:
            print("   ⏭  Skipping (invalid name)")
            continue

        club_id, club_url = find_club(club_name)

        if club_id and club_url:
            df.at[idx, "transfermarkt_id"]  = club_id
            df.at[idx, "transfermarkt_url"] = club_url
            print(f"   ✅ {club_url}")
            found_count += 1
        else:
            failed_clubs.append(club_name)
            print(f"   ❌ Not found")

        # Save progress periodically
        if i % SAVE_EVERY == 0:
            df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
            pct = found_count / i * 100
            print(f"\n  💾 Progress saved ({i} processed, {found_count} found, {pct:.1f}%)\n")

        # Polite delay between clubs
        if i < len(todo_indices):
            delay = random.uniform(*DELAY_RANGE)
            time.sleep(delay)

    # Final save
    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

    # Save clubs still missing
    if failed_clubs:
        pd.DataFrame({"club_name": failed_clubs}).to_csv(MISSING_CSV, index=False)

    # Summary
    total_todo = len(todo_indices)
    print()
    print("=" * 70)
    print("  DONE")
    print(f"  Processed : {total_todo}")
    print(f"  Found     : {found_count}  ({found_count/max(total_todo,1)*100:.1f}%)")
    print(f"  Still missing: {len(failed_clubs)}")
    print(f"  Output    : {OUTPUT_CSV}")
    if failed_clubs:
        print(f"  Missing   : {MISSING_CSV}")
    print("=" * 70)


if __name__ == "__main__":
    run()

  TRANSFERMARKT URL FINDER – IMPROVED
  Loaded 'clubs_missing.csv' (utf-8), 428 rows
  Name column: 'squad'
  Clubs to process: 428 / 428

[1/428] Real Madrid B
   → [full name] no match
   → [short tokens] no match
   → [slug query] no match
   → [parent club]       [reserve] trying parent name: 'Real Madrid'
found! ID=418
   ✅ https://www.transfermarkt.com/real-madrid/startseite/verein/418
[2/428] Sevilla
   → [full name] found! ID=368
   ✅ https://www.transfermarkt.com/fc-sevilla/startseite/verein/368
[3/428] Wolves
   → [full name] found! ID=23106
   ✅ https://www.transfermarkt.com/warri-wolves-fc/startseite/verein/23106
[4/428] Dinamo
   → [full name] found! ID=75231
   ✅ https://www.transfermarkt.com/fk-makhachkala/startseite/verein/75231
[5/428] Roma
   → [full name] found! ID=12
   ✅ https://www.transfermarkt.com/as-rom/startseite/verein/12
[6/428] Montreal Impact
   → [full name] found! ID=79533
   ✅ https://www.transfermarkt.com/montreal-impact-reserves/startseite/verein/7953

In [3]:
"""
SCRIPT: TEST EXTRACTION OF CLUB INFO FROM TRANSFERMARKT
Extracting league, country, market value, squad info, etc.
"""

import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
import re
import time

# ========= CONFIGURATION =========
# Use your file with Transfermarkt URLs
INPUT_CSV = "clubs_with_urls2.csv"  # Your file with transfermarkt_url column
OUTPUT_CSV = "test_clubs_info.csv"

def setup_selenium_driver():
    """Setup Selenium WebDriver"""
    chrome_options = Options()
    chrome_options.add_argument("--headless=new")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--disable-blink-features=AutomationControlled")
    chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
    chrome_options.add_experimental_option('useAutomationExtension', False)
    chrome_options.add_argument("--disable-extensions")
    chrome_options.add_argument("--disable-images")
    chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
    
    driver = webdriver.Chrome(options=chrome_options)
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
    return driver

def extract_club_info_from_transfermarkt(driver, url, club_name):
    """Extract club information from Transfermarkt club page"""
    print(f"🔍 Extracting: {club_name}")
    print(f"   URL: {url}")
    
    club_info = {
        'club_name': club_name,
        'transfermarkt_url': url,
        'official_name': 'Not found',
        'league': 'Not found',
        'league_level': 'Not found',
        'league_country': 'Not found',
        'league_country_flag': 'Not found',
        'table_position': 'Not found',
        'years_in_league': 'Not found',
        'market_value': 'Not found',
        'squad_size': 'Not found',
        'average_age': 'Not found',
        'foreigners': 'Not found',
        'foreigners_percentage': 'Not found',
        'national_team_players': 'Not found',
        'stadium': 'Not found',
        'stadium_capacity': 'Not found',
        'transfer_record': 'Not found',
        'status': 'Success'
    }
    
    try:
        driver.get(url)
        time.sleep(3)
        
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        
        # ========= METHOD 1: Get league from header =========
        # The league is often in a link in the header
        header_club = soup.find('div', class_='data-header__club')
        if header_club:
            # Look for league link
            league_link = header_club.find('a', href=re.compile(r'/wettbewerb/'))
            if league_link:
                league_text = league_link.get_text(strip=True)
                club_info['league'] = league_text
                print(f"   🏆 League: {club_info['league']}")
                
                # Try to extract league level from the same text
                if 'Premier League' in league_text:
                    club_info['league_level'] = 'First Tier England'
                elif 'Championship' in league_text:
                    club_info['league_level'] = 'Second Tier England'
                elif 'League One' in league_text:
                    club_info['league_level'] = 'Third Tier England'
                elif 'LaLiga' in league_text or 'La Liga' in league_text:
                    club_info['league_level'] = 'First Tier Spain'
                elif '2.' in league_text and 'Bundesliga' in league_text:
                    club_info['league_level'] = 'Second Tier Germany'
                elif 'Bundesliga' in league_text:
                    club_info['league_level'] = 'First Tier Germany'
                elif 'Serie A' in league_text:
                    club_info['league_level'] = 'First Tier Italy'
                elif 'Serie B' in league_text:
                    club_info['league_level'] = 'Second Tier Italy'
                elif 'Ligue 1' in league_text:
                    club_info['league_level'] = 'First Tier France'
                elif 'Ligue 2' in league_text:
                    club_info['league_level'] = 'Second Tier France'
                elif 'Eredivisie' in league_text:
                    club_info['league_level'] = 'First Tier Netherlands'
                elif 'Primeira Liga' in league_text:
                    club_info['league_level'] = 'First Tier Portugal'
                elif 'Indian Super League' in league_text:
                    club_info['league_level'] = 'First Tier India'
                    club_info['league_country'] = 'India'
        
        # ========= METHOD 2: Look for the info-table =========
        info_table = soup.find('div', class_='info-table')
        if info_table:
            info_text = info_table.get_text()
            
            # League level (League level: First Tier Spain)
            level_match = re.search(r'League level:\s*([^\n]+)', info_text)
            if level_match:
                club_info['league_level'] = level_match.group(1).strip()
                print(f"   📊 League Level: {club_info['league_level']}")
            
            # Country (often shown with flag)
            country_match = re.search(r'Country:\s*([^\n]+)', info_text)
            if country_match:
                country_text = country_match.group(1).strip()
                # Clean up country name (remove flag emoji if present)
                club_info['league_country'] = re.sub(r'[^\w\s]', '', country_text).strip()
                print(f"   🌍 Country: {club_info['league_country']}")
            
            # Table position
            position_match = re.search(r'Table position:\s*(\d+)', info_text)
            if position_match:
                club_info['table_position'] = position_match.group(1)
                print(f"   📍 Position: {club_info['table_position']}")
            
            # Years in league
            years_match = re.search(r'In league since:\s*([^\n]+)', info_text)
            if years_match:
                club_info['years_in_league'] = years_match.group(1).strip()
                print(f"   📅 Years in league: {club_info['years_in_league']}")
            
            # Stadium
            stadium_match = re.search(r'Stadium:\s*([^,\n]+)', info_text)
            if stadium_match:
                club_info['stadium'] = stadium_match.group(1).strip()
                print(f"   🏟️ Stadium: {club_info['stadium']}")
            
            # Stadium capacity
            capacity_match = re.search(r'(\d{3,}(?:,\d{3})*)\s*seats?', info_text, re.IGNORECASE)
            if capacity_match:
                club_info['stadium_capacity'] = capacity_match.group(1)
                print(f"   👥 Capacity: {club_info['stadium_capacity']}")
        
        # ========= METHOD 3: Look for the data-grid (market values, squad, etc.) =========
        data_grid = soup.find('div', class_='data-grid')
        if data_grid:
            grid_text = data_grid.get_text()
            
            # Market value
            value_match = re.search(r'Total market value:\s*([^€]+€[\d.,]+\s*[mb]?)', grid_text, re.IGNORECASE)
            if not value_match:
                value_match = re.search(r'([€£₹]\s*[\d.,]+\s*[mb]?)', grid_text)
            if value_match:
                club_info['market_value'] = value_match.group(1).strip()
                print(f"   💰 Market Value: {club_info['market_value']}")
            
            # Squad size
            squad_match = re.search(r'Squad size:\s*(\d+)', grid_text)
            if squad_match:
                club_info['squad_size'] = squad_match.group(1)
                print(f"   👥 Squad size: {club_info['squad_size']}")
            
            # Average age
            age_match = re.search(r'Average age:\s*([\d.]+)', grid_text)
            if age_match:
                club_info['average_age'] = age_match.group(1)
                print(f"   🎂 Average age: {club_info['average_age']}")
            
            # Foreigners
            foreign_match = re.search(r'Foreigners:\s*(\d+)\s*\(([\d.]+)%\)', grid_text)
            if foreign_match:
                club_info['foreigners'] = foreign_match.group(1)
                club_info['foreigners_percentage'] = foreign_match.group(2)
                print(f"   🌍 Foreigners: {club_info['foreigners']} ({club_info['foreigners_percentage']}%)")
            
            # National team players
            nt_match = re.search(r'National team players:\s*(\d+)', grid_text)
            if nt_match:
                club_info['national_team_players'] = nt_match.group(1)
                print(f"   🏅 National team players: {club_info['national_team_players']}")
            
            # Transfer record
            transfer_match = re.search(r'Current transfer record:\s*([+-]?[€£₹][\d.,]+\s*[mb]?)', grid_text, re.IGNORECASE)
            if transfer_match:
                club_info['transfer_record'] = transfer_match.group(1)
                print(f"   💸 Transfer record: {club_info['transfer_record']}")
        
        # ========= METHOD 4: Get official name from title =========
        title_elem = soup.find('h1', class_='data-header__headline-wrapper')
        if title_elem:
            official_name = title_elem.get_text(strip=True)
            # Remove trophy count if present
            official_name = re.sub(r'\s+\d+\s*$', '', official_name)
            club_info['official_name'] = official_name
            print(f"   📛 Official Name: {club_info['official_name']}")
        
        return club_info
        
    except Exception as e:
        print(f"   ❌ Error: {str(e)[:100]}")
        club_info['status'] = str(e)[:100]
        return club_info

def test_extraction():
    """Test extraction with a few sample clubs"""
    print("🚀 TEST EXTRACTION FROM TRANSFERMARKT")
    print("=" * 80)
    
    # Sample clubs to test (use clubs that you know have good URLs)
    test_clubs = [
        {"name": "Odisha FC", "url": "https://www.transfermarkt.com/odisha-fc/startseite/verein/12345"},  # Replace with actual URL
        {"name": "Real Madrid", "url": "https://www.transfermarkt.com/real-madrid/startseite/verein/418"},
        {"name": "Sevilla", "url": "https://www.transfermarkt.com/sevilla-fc/startseite/verein/368"},
    ]
    
    driver = setup_selenium_driver()
    all_results = []
    
    try:
        for club in test_clubs:
            print(f"\n{'='*60}")
            info = extract_club_info_from_transfermarkt(driver, club['url'], club['name'])
            all_results.append(info)
            print(f"\n✅ Completed: {club['name']}")
            time.sleep(3)  # Delay between requests
    
    finally:
        driver.quit()
        print("\n✅ WebDriver closed")
    
    # Save results
    df = pd.DataFrame(all_results)
    df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')
    
    print("\n" + "=" * 80)
    print("📊 TEST RESULTS:")
    print("=" * 80)
    
    for info in all_results:
        print(f"\n📌 {info['club_name']}")
        print(f"   Official: {info['official_name']}")
        print(f"   League: {info['league']}")
        print(f"   League Level: {info['league_level']}")
        print(f"   Country: {info['league_country']}")
        print(f"   Position: {info['table_position']}")
        print(f"   Market Value: {info['market_value']}")
        print(f"   Squad: {info['squad_size']} players")
        print(f"   Stadium: {info['stadium']}")
    
    print(f"\n💾 Results saved to: {OUTPUT_CSV}")

def extract_from_csv():
    """Extract info for all clubs in your CSV"""
    print("🚀 EXTRACTING CLUB INFO FROM TRANSFERMARKT")
    print("=" * 80)
    
    try:
        df = pd.read_csv(INPUT_CSV)
        print(f"✅ Loaded {len(df)} clubs")
        
        # Find the URL column
        url_column = None
        for col in ['transfermarkt_url', 'url', 'fbref_url', 'club_url']:
            if col in df.columns:
                url_column = col
                break
        
        if not url_column:
            print(f"📋 Available columns: {list(df.columns)}")
            print("Please ensure your CSV has a column with Transfermarkt URLs")
            return
        
        # Find club name column
        name_column = df.columns[0]  # Assume first column has names
        
        print(f"📊 Using '{name_column}' for club names")
        print(f"📊 Using '{url_column}' for URLs")
        
        # Filter clubs with valid URLs
        valid_clubs = df[df[url_column].notna() & (df[url_column] != 'Not found')]
        print(f"🔧 Clubs with valid URLs: {len(valid_clubs)}")
        
        driver = setup_selenium_driver()
        all_results = []
        
        try:
            for idx, row in valid_clubs.iterrows():
                club_name = str(row[name_column]).strip()
                club_url = str(row[url_column]).strip()
                
                print(f"\n[{idx + 1}/{len(valid_clubs)}] {club_name}")
                
                info = extract_club_info_from_transfermarkt(driver, club_url, club_name)
                all_results.append(info)
                
                # Save progress every 10 clubs
                if len(all_results) % 10 == 0:
                    temp_df = pd.DataFrame(all_results)
                    temp_df.to_csv(f"progress_{len(all_results)}.csv", index=False)
                    print(f"💾 Progress saved: {len(all_results)} clubs")
                
                time.sleep(random.uniform(2, 4))
        
        finally:
            driver.quit()
        
        # Save all results
        df_results = pd.DataFrame(all_results)
        df_results.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')
        
        print("\n" + "=" * 80)
        print("🎉 EXTRACTION COMPLETED!")
        print("=" * 80)
        print(f"📊 Total clubs processed: {len(all_results)}")
        print(f"💾 Results saved to: {OUTPUT_CSV}")
        
    except FileNotFoundError:
        print(f"❌ File '{INPUT_CSV}' not found")
    except Exception as e:
        print(f"❌ Error: {e}")

if __name__ == "__main__":
    print("Choose an option:")
    print("1. Test with sample clubs (Real Madrid, Sevilla)")
    print("2. Extract from all clubs in CSV")
    
    choice = input("Enter 1 or 2: ").strip()
    
    if choice == "1":
        test_extraction()
    else:
        extract_from_csv()

Choose an option:
1. Test with sample clubs (Real Madrid, Sevilla)
2. Extract from all clubs in CSV
🚀 TEST EXTRACTION FROM TRANSFERMARKT

🔍 Extracting: Odisha FC
   URL: https://www.transfermarkt.com/odisha-fc/startseite/verein/12345
   📛 Official Name: Erzincanspor

✅ Completed: Odisha FC

🔍 Extracting: Real Madrid
   URL: https://www.transfermarkt.com/real-madrid/startseite/verein/418
   📛 Official Name: Real Madrid

✅ Completed: Real Madrid

🔍 Extracting: Sevilla
   URL: https://www.transfermarkt.com/sevilla-fc/startseite/verein/368
   📛 Official Name: Sevilla FC

✅ Completed: Sevilla

✅ WebDriver closed

📊 TEST RESULTS:

📌 Odisha FC
   Official: Erzincanspor
   League: Not found
   League Level: Not found
   Country: Not found
   Position: Not found
   Market Value: Not found
   Squad: Not found players
   Stadium: Not found

📌 Real Madrid
   Official: Real Madrid
   League: Not found
   League Level: Not found
   Country: Not found
   Position: Not found
   Market Value: Not found

In [8]:
"""
TRANSFERMARKT CLUB INFO SCRAPER – v3 (selectors confirmed from debug output)

Extracts: official_name, league, league_code, league_level, league_country,
          table_position, years_in_league, market_value, squad_size,
          average_age, foreigners, foreigners_pct, national_team_players,
          stadium, stadium_capacity, transfer_record

Usage:
    python scrape_transfermarkt.py
    → 1: debug one URL
    → 2: test 4 known clubs
    → 3: full CSV run
"""

import re
import time
import random

import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# ── CONFIG ───────────────────────────────────────────────────────────────────
INPUT_CSV   = "clubs_with_urls2.csv"   # must have: squad (or club_name), transfermarkt_url
OUTPUT_CSV  = "clubs_info2.csv"
SAVE_EVERY  = 15
PAGE_WAIT   = 5                   # seconds after driver.get()

# Wettbewerb code → country  (extend as needed)
LEAGUE_CODE_TO_COUNTRY = {
    "GB1": "England",    "GB2": "England",    "GB3": "England",   "GB4": "England",
    "ES1": "Spain",      "ES2": "Spain",      "ES3": "Spain",
    "L1":  "Germany",    "L2":  "Germany",    "L3":  "Germany",
    "IT1": "Italy",      "IT2": "Italy",
    "FR1": "France",     "FR2": "France",
    "NL1": "Netherlands","BE1": "Belgium",    "PT1": "Portugal",
    "TR1": "Turkey",     "TR2": "Turkey",     "TR3A": "Turkey",   "TR3B": "Turkey",
    "RU1": "Russia",     "UKR1": "Ukraine",   "SC1": "Scotland",
    "GR1": "Greece",     "DK1": "Denmark",    "SE1": "Sweden",    "NO1": "Norway",
    "SW1": "Switzerland","A1":  "Austria",    "PO1": "Poland",    "CZ1": "Czech Republic",
    "RO1": "Romania",    "SK1": "Slovakia",   "HU1": "Hungary",   "CRO1": "Croatia",
    "SRB": "Serbia",     "BUL1": "Bulgaria",  "SVN1": "Slovenia",
    "US1": "USA",        "MX1": "Mexico",     "BRA1": "Brazil",   "ARG1": "Argentina",
    "COL1": "Colombia",  "CHL1": "Chile",     "PER1": "Peru",     "URU1": "Uruguay",
    "JAP1": "Japan",     "KOR1": "South Korea","CHN1": "China",   "AUS1": "Australia",
    "IND1": "India",     "SAU1": "Saudi Arabia","UAE": "UAE",
    "EGY1": "Egypt",     "MAR1": "Morocco",   "RSA1": "South Africa",
    "CL":  "Europe",     "EL":  "Europe",     "UCOL": "Europe",
}
# ─────────────────────────────────────────────────────────────────────────────


# ── DRIVER ───────────────────────────────────────────────────────────────────

def make_driver():
    opts = Options()
    opts.add_argument("--headless=new")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--disable-blink-features=AutomationControlled")
    opts.add_experimental_option("excludeSwitches", ["enable-automation"])
    opts.add_experimental_option("useAutomationExtension", False)
    opts.add_argument("--window-size=1920,1080")
    opts.add_argument("--lang=en-US")
    opts.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
    )
    driver = webdriver.Chrome(options=opts)
    driver.execute_script(
        "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
    )
    return driver


def get_soup(driver, url: str) -> BeautifulSoup:
    driver.get(url)
    try:
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "div.data-header"))
        )
    except Exception:
        pass
    time.sleep(PAGE_WAIT)
    return BeautifulSoup(driver.page_source, "lxml")


# ── EXTRACTORS ────────────────────────────────────────────────────────────────

def _get_league_code(href: str) -> str:
    m = re.search(r"/wettbewerb/([A-Z0-9]+)", href, re.I)
    return m.group(1).upper() if m else ""


def extract(driver, url: str, club_name: str) -> dict:
    result = {
        "club_name":             club_name,
        "transfermarkt_url":     url,
        "official_name":         "",
        "league":                "",
        "league_code":           "",
        "league_url":            "",
        "league_level":          "",
        "league_country":        "",
        "table_position":        "",
        "years_in_league":       "",
        "market_value":          "",
        "squad_size":            "",
        "average_age":           "",
        "foreigners":            "",
        "foreigners_pct":        "",
        "national_team_players": "",
        "stadium":               "",
        "stadium_capacity":      "",
        "transfer_record":       "",
        "status":                "ok",
    }

    try:
        soup = get_soup(driver, url)
        text = soup.get_text(separator="\n")

        # ── Official name from <title> ─────────────────────────────────────
        # "<ClubName> - Club profile | Transfermarkt"
        title_tag = soup.find("title")
        if title_tag:
            m = re.match(r"^(.+?)\s*[-–|]", title_tag.get_text())
            if m:
                result["official_name"] = m.group(1).strip()

        # ── League: first wettbewerb link AFTER the h1 club name ──────────
        # The top nav also has /wettbewerb/ links (CL, Premier League, etc.)
        # but they appear BEFORE the h1. We skip those.
        h1 = soup.find("h1")
        league_link = None
        if h1:
            for a in h1.find_all_next("a", href=re.compile(r"/wettbewerb/")):
                href = a.get("href", "")
                txt  = a.get_text(strip=True)
                # Skip blank, pure-number (position), and matchday links
                if txt and not txt.isdigit() and "/spieltag/" not in href:
                    league_link = a
                    break

        if league_link:
            href = league_link.get("href", "")
            result["league"]      = league_link.get_text(strip=True)
            result["league_code"] = _get_league_code(href)
            result["league_url"]  = (
                href if href.startswith("http")
                else "https://www.transfermarkt.com" + href
            )
            result["league_country"] = LEAGUE_CODE_TO_COUNTRY.get(
                result["league_code"], ""
            )

        # ── League level: link immediately after the league link ───────────
        # Transfermarkt renders: [league link] → [tier link] → [position link]
        # e.g. "2.Lig Beyaz" → "Third Tier" → "10"
        if league_link:
            for a in league_link.find_all_next("a", href=re.compile(r"/wettbewerb/")):
                txt = a.get_text(strip=True)
                if re.match(r"(First|Second|Third|Fourth|Fifth|Sixth)\s+Tier", txt, re.I):
                    result["league_level"] = txt
                    break

        # Fallback: regex on page text
        if not result["league_level"]:
            m = re.search(r"League level:\s*\n?\s*((First|Second|Third|Fourth|Fifth|Sixth)\s+Tier)", text, re.I)
            if m:
                result["league_level"] = m.group(1).strip()

        # ── Table position ────────────────────────────────────────────────
        m = re.search(r"Table position:\s*\n?\s*(\d+)", text)
        if m:
            result["table_position"] = m.group(1)

        # ── Years in league ───────────────────────────────────────────────
        m = re.search(r"In league since:\s*\n?\s*(\d+\s*years?)", text, re.I)
        if m:
            result["years_in_league"] = m.group(1).strip()

        # ── Market value ──────────────────────────────────────────────────
        # The page text renders the value split across lines:
        #   "€\n2.37\nm \n\nTotal market value"
        # Pattern A: symbol, number, suffix all on separate lines near "Total market value"
        m = re.search(
            r"([€£$₹])\s*\n\s*([\d.,]+)\s*\n\s*([mkbMKB])\s*\n[^\n]*Total market value",
            text
        )
        if m:
            result["market_value"] = f"{m.group(1)}{m.group(2)}{m.group(3)}"

        # Pattern B: label first, then value
        if not result["market_value"]:
            m = re.search(
                r"Total market value[^\n]*\n[^\n]*\n?\s*([€£$₹]\s*[\d.,]+\s*[mkbMKB]?)",
                text
            )
            if m:
                result["market_value"] = m.group(1).strip()

        # Pattern C: compact inline
        if not result["market_value"]:
            m = re.search(r"([€£$₹][\d.,]+\s*[mkbMKB])", text)
            if m:
                result["market_value"] = m.group(1).strip()

        # ── Squad size ────────────────────────────────────────────────────
        m = re.search(r"Squad size:\s*\n?\s*(\d+)", text)
        if m:
            result["squad_size"] = m.group(1)

        # ── Average age ───────────────────────────────────────────────────
        m = re.search(r"Average age:\s*\n?\s*([\d.]+)", text)
        if m:
            result["average_age"] = m.group(1)

        # ── Foreigners ────────────────────────────────────────────────────
        # "Foreigners:\n0\n  \n %"  or  "Foreigners:\n12\n(52.2%)"
        m = re.search(r"Foreigners:\s*\n?\s*(\d+)[^\d\n]*([\d.]+)\s*%", text)
        if m:
            result["foreigners"]     = m.group(1)
            result["foreigners_pct"] = m.group(2)
        else:
            m = re.search(r"Foreigners:\s*\n?\s*(\d+)", text)
            if m:
                result["foreigners"] = m.group(1)

        # ── National team players ─────────────────────────────────────────
        m = re.search(r"National team players:\s*\n?\s*(\d+)", text)
        if m:
            result["national_team_players"] = m.group(1)

        # ── Stadium + capacity ────────────────────────────────────────────
        # "Stadium:\nStadium Name\n  \n80,000 Seats"
        m = re.search(
            r"Stadium:\s*\n\s*([^\n]+)\n[^\n]*([\d,]+)\s*Seats?",
            text, re.I
        )
        if m:
            result["stadium"]          = m.group(1).strip()
            result["stadium_capacity"] = m.group(2).replace(",", "")
        else:
            m = re.search(r"Stadium:\s*\n\s*([^\n]+)", text)
            if m:
                result["stadium"] = m.group(1).strip()

        # ── Transfer record ───────────────────────────────────────────────
        # "Current transfer record:\n+-0"  or  "+€45.00m"
        m = re.search(
            r"Current transfer record:\s*\n?\s*([+\-±]?[€£$₹]?[\d.,]+[mkbMKB]?)",
            text, re.I
        )
        if m:
            result["transfer_record"] = m.group(1).strip()

    except Exception as e:
        result["status"] = str(e)[:150]

    return result


# ── DEBUG ─────────────────────────────────────────────────────────────────────

def debug_url(driver, url: str):
    print(f"\nFetching: {url}\n")
    soup = get_soup(driver, url)
    r    = extract(driver, url, "DEBUG")

    print("=" * 60)
    print("EXTRACTED FIELDS:")
    for k, v in r.items():
        icon = "✅" if (v and k != "status") else ("✅" if k == "status" and v == "ok" else "❌")
        print(f"  {icon}  {k:28s} {v}")

    print("\n" + "=" * 60)
    h1 = soup.find("h1")
    print("Wettbewerb links AFTER h1 (first 10):")
    if h1:
        for a in list(h1.find_all_next("a", href=re.compile(r"/wettbewerb/")))[:10]:
            print(f"  '{a.get_text(strip=True):40s}'  {a.get('href','')}")


# ── TEST ──────────────────────────────────────────────────────────────────────

TEST_CLUBS = [
    {"name": "Real Madrid",     "url": "https://www.transfermarkt.com/real-madrid/startseite/verein/418"},
    {"name": "Sevilla FC",      "url": "https://www.transfermarkt.com/sevilla-fc/startseite/verein/368"},
    {"name": "Manchester City", "url": "https://www.transfermarkt.com/manchester-city/startseite/verein/281"},
    {"name": "Ankaraspor",      "url": "https://www.transfermarkt.com/ankaraspor/startseite/verein/2944"},
]

def run_test(driver):
    print("\n🧪 TEST MODE")
    print("=" * 70)
    results = []
    for club in TEST_CLUBS:
        print(f"\n→ {club['name']}")
        r = extract(driver, club["url"], club["name"])
        results.append(r)
        for k, v in r.items():
            if k not in ("club_name", "transfermarkt_url"):
                icon = "✅" if v and k != "status" else ""
                print(f"   {icon}  {k:28s} {v}")
        time.sleep(random.uniform(3, 5))

    df = pd.DataFrame(results)
    df.to_csv("test_clubs_info.csv", index=False)
    print("\n💾 Saved → test_clubs_info.csv")
    cols = ["club_name", "league", "league_level", "league_country",
            "market_value", "squad_size", "average_age"]
    print(df[cols].to_string(index=False))


# ── FULL CSV ──────────────────────────────────────────────────────────────────

def run_csv(driver):
    print(f"\n📂 Loading {INPUT_CSV} …")
    df = None
    for enc in ["utf-8", "latin-1", "cp1252"]:
        try:
            df = pd.read_csv(INPUT_CSV, encoding=enc)
            break
        except Exception:
            continue
    if df is None:
        print("ERROR: could not read CSV")
        return

    name_col = next(
        (c for c in ["squad", "club_name", "name", "Club"] if c in df.columns),
        df.columns[0]
    )
    url_col = next(
        (c for c in ["transfermarkt_url", "url"] if c in df.columns), None
    )
    if not url_col:
        print("ERROR: no URL column found (expected 'transfermarkt_url')")
        return

    valid = df[df[url_col].astype(str).str.startswith("https://")].copy()
    print(f"  Valid URLs: {len(valid)} / {len(df)}")

    results = []
    for i, (_, row) in enumerate(valid.iterrows(), 1):
        name = str(row[name_col]).strip()
        url  = str(row[url_col]).strip()
        print(f"\n[{i}/{len(valid)}] {name}")

        r = extract(driver, url, name)
        results.append(r)
        print(
            f"   league={r['league'] or '—'} | "
            f"level={r['league_level'] or '—'} | "
            f"country={r['league_country'] or '—'} | "
            f"value={r['market_value'] or '—'} | "
            f"status={r['status']}"
        )

        if i % SAVE_EVERY == 0:
            pd.DataFrame(results).to_csv(OUTPUT_CSV, index=False)
            print(f"   💾 Progress saved ({i} clubs)")

        time.sleep(random.uniform(3, 6))

    pd.DataFrame(results).to_csv(OUTPUT_CSV, index=False)
    print(f"\n✅ Done! {len(results)} clubs → {OUTPUT_CSV}")


# ── ENTRY POINT ───────────────────────────────────────────────────────────────

if __name__ == "__main__":
    print("=" * 60)
    print("  TRANSFERMARKT SCRAPER  v3")
    print("=" * 60)
    print("1. Debug one URL")
    print("2. Test 4 clubs (Real Madrid, Sevilla, Man City, Ankaraspor)")
    print("3. Full CSV run")
    choice = input("\nEnter 1 / 2 / 3: ").strip()

    driver = make_driver()
    try:
        if choice == "1":
            url = input("Paste Transfermarkt URL: ").strip()
            debug_url(driver, url)
        elif choice == "2":
            run_test(driver)
        else:
            run_csv(driver)
    finally:
        driver.quit()
        print("\n✅ Driver closed.")

  TRANSFERMARKT SCRAPER  v3
1. Debug one URL
2. Test 4 clubs (Real Madrid, Sevilla, Man City, Ankaraspor)
3. Full CSV run

📂 Loading clubs_with_urls2.csv …
  Valid URLs: 1218 / 1218

[1/1218] Real Madrid B
   league=Primera Federación - Grupo I | level=Third Tier | country=— | value=€62.00m | status=ok

[2/1218] AlmerÃ­a
   league=Segunda División | level=— | country=Spain | value=— | status=ok

[3/1218] Sevilla
   league=LaLiga | level=First Tier | country=Spain | value=€140.40m | status=ok

[4/1218] Manchester City
   league=Premier League | level=First Tier | country=England | value=€45.00m | status=ok

[5/1218] Valencia
   league=LaLiga | level=— | country=Spain | value=— | status=ok

[6/1218] Middlesbrough
   league=Championship | level=Second Tier | country=England | value=€128.90m | status=ok

[7/1218] BeÅŸiktaÅŸ
   league=Süper Lig | level=First Tier | country=Turkey | value=€185.10m | status=ok

[8/1218] CÃ¡diz
   league=Segunda División | level=— | country=Spain | value=— | s

In [13]:
"""
TRANSFERMARKT CLUB INFO SCRAPER – v3 (selectors confirmed from debug output)

Extracts: official_name, league, league_code, league_level, league_country,
          table_position, years_in_league, market_value, squad_size,
          average_age, foreigners, foreigners_pct, national_team_players,
          stadium, stadium_capacity, transfer_record

Usage:
    python scrape_transfermarkt.py
    → 1: debug one URL
    → 2: test 4 known clubs
    → 3: full CSV run
"""

import re
import time
import random

import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# ── CONFIG ───────────────────────────────────────────────────────────────────
INPUT_CSV   = "clubs_with_urls4.csv"   # must have: squad (or club_name), transfermarkt_url
OUTPUT_CSV  = "clubs_info5.csv"
SAVE_EVERY  = 15
PAGE_WAIT   = 5                   # seconds after driver.get()

# Wettbewerb code → country  (extend as needed)
LEAGUE_CODE_TO_COUNTRY = {
    "GB1": "England",    "GB2": "England",    "GB3": "England",   "GB4": "England",
    "ES1": "Spain",      "ES2": "Spain",      "ES3": "Spain",
    "L1":  "Germany",    "L2":  "Germany",    "L3":  "Germany",
    "IT1": "Italy",      "IT2": "Italy",
    "FR1": "France",     "FR2": "France",
    "NL1": "Netherlands","BE1": "Belgium",    "PT1": "Portugal",
    "TR1": "Turkey",     "TR2": "Turkey",     "TR3A": "Turkey",   "TR3B": "Turkey",
    "RU1": "Russia",     "UKR1": "Ukraine",   "SC1": "Scotland",
    "GR1": "Greece",     "DK1": "Denmark",    "SE1": "Sweden",    "NO1": "Norway",
    "SW1": "Switzerland","A1":  "Austria",    "PO1": "Poland",    "CZ1": "Czech Republic",
    "RO1": "Romania",    "SK1": "Slovakia",   "HU1": "Hungary",   "CRO1": "Croatia",
    "SRB": "Serbia",     "BUL1": "Bulgaria",  "SVN1": "Slovenia",
    "US1": "USA",        "MX1": "Mexico",     "BRA1": "Brazil",   "ARG1": "Argentina",
    "COL1": "Colombia",  "CHL1": "Chile",     "PER1": "Peru",     "URU1": "Uruguay",
    "JAP1": "Japan",     "KOR1": "South Korea","CHN1": "China",   "AUS1": "Australia",
    "IND1": "India",     "SAU1": "Saudi Arabia","UAE": "UAE",
    "EGY1": "Egypt",     "MAR1": "Morocco",   "RSA1": "South Africa",
    "CL":  "Europe",     "EL":  "Europe",     "UCOL": "Europe",
}
# ─────────────────────────────────────────────────────────────────────────────


# ── DRIVER ───────────────────────────────────────────────────────────────────

def make_driver():
    opts = Options()
    opts.add_argument("--headless=new")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--disable-blink-features=AutomationControlled")
    opts.add_experimental_option("excludeSwitches", ["enable-automation"])
    opts.add_experimental_option("useAutomationExtension", False)
    opts.add_argument("--window-size=1920,1080")
    opts.add_argument("--lang=en-US")
    opts.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
    )
    driver = webdriver.Chrome(options=opts)
    driver.execute_script(
        "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
    )
    return driver


def get_soup(driver, url: str) -> BeautifulSoup:
    driver.get(url)
    try:
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "div.data-header"))
        )
    except Exception:
        pass
    time.sleep(PAGE_WAIT)
    return BeautifulSoup(driver.page_source, "lxml")


# ── EXTRACTORS ────────────────────────────────────────────────────────────────

def _get_league_code(href: str) -> str:
    m = re.search(r"/wettbewerb/([A-Z0-9]+)", href, re.I)
    return m.group(1).upper() if m else ""


def extract(driver, url: str, club_name: str) -> dict:
    result = {
        "club_name":             club_name,
        "transfermarkt_url":     url,
        "official_name":         "",
        "league":                "",
        "league_code":           "",
        "league_url":            "",
        "league_level":          "",
        "league_country":        "",
        "table_position":        "",
        "years_in_league":       "",
        "market_value":          "",
        "squad_size":            "",
        "average_age":           "",
        "foreigners":            "",
        "foreigners_pct":        "",
        "national_team_players": "",
        "stadium":               "",
        "stadium_capacity":      "",
        "transfer_record":       "",
        "status":                "ok",
    }

    try:
        soup = get_soup(driver, url)
        text = soup.get_text(separator="\n")

        # ── Official name from <title> ─────────────────────────────────────
        # "<ClubName> - Club profile | Transfermarkt"
        title_tag = soup.find("title")
        if title_tag:
            m = re.match(r"^(.+?)\s*[-–|]", title_tag.get_text())
            if m:
                result["official_name"] = m.group(1).strip()

        # ── League: first wettbewerb link AFTER the h1 club name ──────────
        # The top nav also has /wettbewerb/ links (CL, Premier League, etc.)
        # but they appear BEFORE the h1. We skip those.
        h1 = soup.find("h1")
        league_link = None
        if h1:
            for a in h1.find_all_next("a", href=re.compile(r"/wettbewerb/")):
                href = a.get("href", "")
                txt  = a.get_text(strip=True)
                # Skip blank, pure-number (position), and matchday links
                if txt and not txt.isdigit() and "/spieltag/" not in href:
                    league_link = a
                    break

        if league_link:
            href = league_link.get("href", "")
            result["league"]      = league_link.get_text(strip=True)
            result["league_code"] = _get_league_code(href)
            result["league_url"]  = (
                href if href.startswith("http")
                else "https://www.transfermarkt.com" + href
            )
            result["league_country"] = LEAGUE_CODE_TO_COUNTRY.get(
                result["league_code"], ""
            )

        # ── League level: link immediately after the league link ───────────
        # Transfermarkt renders: [league link] → [tier link] → [position link]
        # e.g. "2.Lig Beyaz" → "Third Tier" → "10"
        if league_link:
            for a in league_link.find_all_next("a", href=re.compile(r"/wettbewerb/")):
                txt = a.get_text(strip=True)
                if re.match(r"(First|Second|Third|Fourth|Fifth|Sixth)\s+Tier", txt, re.I):
                    result["league_level"] = txt
                    break

        # Fallback: regex on page text
        if not result["league_level"]:
            m = re.search(r"League level:\s*\n?\s*((First|Second|Third|Fourth|Fifth|Sixth)\s+Tier)", text, re.I)
            if m:
                result["league_level"] = m.group(1).strip()

        # ── Table position ────────────────────────────────────────────────
        m = re.search(r"Table position:\s*\n?\s*(\d+)", text)
        if m:
            result["table_position"] = m.group(1)

        # ── Years in league ───────────────────────────────────────────────
        m = re.search(r"In league since:\s*\n?\s*(\d+\s*years?)", text, re.I)
        if m:
            result["years_in_league"] = m.group(1).strip()

        # ── Market value ──────────────────────────────────────────────────
        # The page text renders the value split across lines:
        #   "€\n2.37\nm \n\nTotal market value"
        # Pattern A: symbol, number, suffix all on separate lines near "Total market value"
        m = re.search(
            r"([€£$₹])\s*\n\s*([\d.,]+)\s*\n\s*([mkbMKB])\s*\n[^\n]*Total market value",
            text
        )
        if m:
            result["market_value"] = f"{m.group(1)}{m.group(2)}{m.group(3)}"

        # Pattern B: label first, then value
        if not result["market_value"]:
            m = re.search(
                r"Total market value[^\n]*\n[^\n]*\n?\s*([€£$₹]\s*[\d.,]+\s*[mkbMKB]?)",
                text
            )
            if m:
                result["market_value"] = m.group(1).strip()

        # Pattern C: compact inline
        if not result["market_value"]:
            m = re.search(r"([€£$₹][\d.,]+\s*[mkbMKB])", text)
            if m:
                result["market_value"] = m.group(1).strip()

        # ── Squad size ────────────────────────────────────────────────────
        m = re.search(r"Squad size:\s*\n?\s*(\d+)", text)
        if m:
            result["squad_size"] = m.group(1)

        # ── Average age ───────────────────────────────────────────────────
        m = re.search(r"Average age:\s*\n?\s*([\d.]+)", text)
        if m:
            result["average_age"] = m.group(1)

        # ── Foreigners ────────────────────────────────────────────────────
        # "Foreigners:\n0\n  \n %"  or  "Foreigners:\n12\n(52.2%)"
        m = re.search(r"Foreigners:\s*\n?\s*(\d+)[^\d\n]*([\d.]+)\s*%", text)
        if m:
            result["foreigners"]     = m.group(1)
            result["foreigners_pct"] = m.group(2)
        else:
            m = re.search(r"Foreigners:\s*\n?\s*(\d+)", text)
            if m:
                result["foreigners"] = m.group(1)

        # ── National team players ─────────────────────────────────────────
        m = re.search(r"National team players:\s*\n?\s*(\d+)", text)
        if m:
            result["national_team_players"] = m.group(1)

        # ── Stadium + capacity ────────────────────────────────────────────
        # "Stadium:\nStadium Name\n  \n80,000 Seats"
        m = re.search(
            r"Stadium:\s*\n\s*([^\n]+)\n[^\n]*([\d,]+)\s*Seats?",
            text, re.I
        )
        if m:
            result["stadium"]          = m.group(1).strip()
            result["stadium_capacity"] = m.group(2).replace(",", "")
        else:
            m = re.search(r"Stadium:\s*\n\s*([^\n]+)", text)
            if m:
                result["stadium"] = m.group(1).strip()

        # ── Transfer record ───────────────────────────────────────────────
        # "Current transfer record:\n+-0"  or  "+€45.00m"
        m = re.search(
            r"Current transfer record:\s*\n?\s*([+\-±]?[€£$₹]?[\d.,]+[mkbMKB]?)",
            text, re.I
        )
        if m:
            result["transfer_record"] = m.group(1).strip()

    except Exception as e:
        result["status"] = str(e)[:150]

    return result


# ── DEBUG ─────────────────────────────────────────────────────────────────────

def debug_url(driver, url: str):
    print(f"\nFetching: {url}\n")
    soup = get_soup(driver, url)
    r    = extract(driver, url, "DEBUG")

    print("=" * 60)
    print("EXTRACTED FIELDS:")
    for k, v in r.items():
        icon = "✅" if (v and k != "status") else ("✅" if k == "status" and v == "ok" else "❌")
        print(f"  {icon}  {k:28s} {v}")

    print("\n" + "=" * 60)
    h1 = soup.find("h1")
    print("Wettbewerb links AFTER h1 (first 10):")
    if h1:
        for a in list(h1.find_all_next("a", href=re.compile(r"/wettbewerb/")))[:10]:
            print(f"  '{a.get_text(strip=True):40s}'  {a.get('href','')}")


# ── TEST ──────────────────────────────────────────────────────────────────────

TEST_CLUBS = [
    {"name": "Real Madrid",     "url": "https://www.transfermarkt.com/real-madrid/startseite/verein/418"},
    {"name": "Sevilla FC",      "url": "https://www.transfermarkt.com/sevilla-fc/startseite/verein/368"},
    {"name": "Manchester City", "url": "https://www.transfermarkt.com/manchester-city/startseite/verein/281"},
    {"name": "Ankaraspor",      "url": "https://www.transfermarkt.com/ankaraspor/startseite/verein/2944"},
]

def run_test(driver):
    print("\n🧪 TEST MODE")
    print("=" * 70)
    results = []
    for club in TEST_CLUBS:
        print(f"\n→ {club['name']}")
        r = extract(driver, club["url"], club["name"])
        results.append(r)
        for k, v in r.items():
            if k not in ("club_name", "transfermarkt_url"):
                icon = "✅" if v and k != "status" else ""
                print(f"   {icon}  {k:28s} {v}")
        time.sleep(random.uniform(3, 5))

    df = pd.DataFrame(results)
    df.to_csv("test_clubs_info.csv", index=False)
    print("\n💾 Saved → test_clubs_info.csv")
    cols = ["club_name", "league", "league_level", "league_country",
            "market_value", "squad_size", "average_age"]
    print(df[cols].to_string(index=False))


# ── FULL CSV ──────────────────────────────────────────────────────────────────

def load_csv_safe(path: str) -> pd.DataFrame | None:
    for enc in ["utf-8", "latin-1", "cp1252"]:
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception:
            continue
    return None


def run_csv(driver):
    print(f"\n📂 Loading {INPUT_CSV} …")
    df = load_csv_safe(INPUT_CSV)
    if df is None:
        print("ERROR: could not read input CSV")
        return

    name_col = next(
        (c for c in ["squad", "club_name", "name", "Club"] if c in df.columns),
        df.columns[0]
    )
    url_col = next(
        (c for c in ["transfermarkt_url", "url"] if c in df.columns), None
    )
    if not url_col:
        print("ERROR: no URL column found (expected 'transfermarkt_url')")
        return

    valid = df[df[url_col].astype(str).str.startswith("https://")].copy()
    total = len(valid)
    print(f"  Valid URLs: {total} / {len(df)}")

    # ── Resume: load already-scraped clubs from OUTPUT_CSV ────────────────
    already_done: set[str] = set()
    existing_rows: list[dict] = []

    import os
    if os.path.exists(OUTPUT_CSV):
        done_df = load_csv_safe(OUTPUT_CSV)
        if done_df is not None and "transfermarkt_url" in done_df.columns:
            # Only count rows where scraping actually succeeded (status == ok)
            ok_rows = done_df[done_df.get("status", "ok") == "ok"] if "status" in done_df.columns else done_df
            already_done = set(ok_rows["transfermarkt_url"].astype(str).tolist())
            existing_rows = done_df.to_dict("records")
            print(f"  Resuming — {len(already_done)} clubs already scraped, skipping them.")

    results: list[dict] = list(existing_rows)   # start with what we already have
    consecutive_errors  = 0
    MAX_CONSECUTIVE     = 3   # restart driver after this many crashes in a row

    for i, (_, row) in enumerate(valid.iterrows(), 1):
        name = str(row[name_col]).strip()
        url  = str(row[url_col]).strip()

        # Skip if already scraped
        if url in already_done:
            print(f"[{i}/{total}] ⏭  {name}  (already done)")
            continue

        print(f"\n[{i}/{total}] {name}")

        # ── Try to scrape; restart driver on session crash ─────────────
        r = None
        for attempt in range(1, 4):   # up to 3 attempts per club
            try:
                r = extract(driver, url, name)
                consecutive_errors = 0
                break
            except Exception as e:
                err = str(e)
                print(f"   ⚠️  Attempt {attempt} failed: {err[:80]}")

                # "Invalid session id" or any WebDriver crash → restart driver
                if any(kw in err.lower() for kw in
                       ["invalid session", "session deleted", "no such session",
                        "chrome not reachable", "disconnected"]):
                    print("   🔄 Restarting Chrome driver …")
                    try:
                        driver.quit()
                    except Exception:
                        pass
                    time.sleep(5)
                    driver = make_driver()
                    consecutive_errors += 1
                else:
                    time.sleep(3)

                if attempt == 3:
                    r = {
                        "club_name":         name,
                        "transfermarkt_url": url,
                        "status":            f"failed after 3 attempts: {err[:80]}",
                    }

        if r is None:
            r = {"club_name": name, "transfermarkt_url": url, "status": "unknown error"}

        results.append(r)
        print(
            f"   league={r.get('league') or '—'} | "
            f"level={r.get('league_level') or '—'} | "
            f"country={r.get('league_country') or '—'} | "
            f"value={r.get('market_value') or '—'} | "
            f"status={r.get('status', '?')}"
        )

        # ── Save progress every SAVE_EVERY clubs ──────────────────────
        scraped_so_far = i - len(already_done)   # approximate
        if scraped_so_far > 0 and scraped_so_far % SAVE_EVERY == 0:
            pd.DataFrame(results).to_csv(OUTPUT_CSV, index=False)
            print(f"   💾 Progress saved ({len(results)} total rows in file)")

        # If too many crashes in a row, pause longer
        if consecutive_errors >= MAX_CONSECUTIVE:
            print(f"   ⏳ {consecutive_errors} crashes in a row — waiting 60s before continuing …")
            time.sleep(60)
            consecutive_errors = 0

        time.sleep(random.uniform(3, 6))

    # Final save
    pd.DataFrame(results).to_csv(OUTPUT_CSV, index=False)
    scraped_new = len(results) - len(existing_rows)
    print(f"\n✅ Done!  {scraped_new} new + {len(existing_rows)} previous = {len(results)} total → {OUTPUT_CSV}")


# ── ENTRY POINT ───────────────────────────────────────────────────────────────

if __name__ == "__main__":
    print("=" * 60)
    print("  TRANSFERMARKT SCRAPER  v3")
    print("=" * 60)
    print("1. Debug one URL")
    print("2. Test 4 clubs (Real Madrid, Sevilla, Man City, Ankaraspor)")
    print("3. Full CSV run")
    choice = input("\nEnter 1 / 2 / 3: ").strip()

    driver = make_driver()
    try:
        if choice == "1":
            url = input("Paste Transfermarkt URL: ").strip()
            debug_url(driver, url)
        elif choice == "2":
            run_test(driver)
        else:
            run_csv(driver)
    finally:
        driver.quit()
        print("\n✅ Driver closed.")

  TRANSFERMARKT SCRAPER  v3
1. Debug one URL
2. Test 4 clubs (Real Madrid, Sevilla, Man City, Ankaraspor)
3. Full CSV run

📂 Loading clubs_with_urls4.csv …
  Valid URLs: 133 / 133

[1/133] Brighton
   league=Premier League | level=First Tier | country=England | value=€494.00m | status=ok

[2/133] Newell's OB
   league=Torneo Apertura | level=— | country=Argentina | value=— | status=ok

[3/133] Reus
   league=Segunda Federación - Grupo III | level=Fourth Tier | country=— | value=€1.65m | status=ok

[4/133] AD Cali
   league=Liga DIMAYOR Apertura | level=— | country=— | value=— | status=ok

[5/133] ADU Magdalena
   league=Torneo DIMAYOR I | level=— | country=— | value=— | status=ok

[6/133] FamalicÃ£o
   league=Liga Portugal | level=— | country=Poland | value=— | status=ok

[7/133] Ath Paranaense
   league=Campeonato Brasileiro Série A | level=— | country=Brazil | value=— | status=ok

[8/133] Kawa Frontale
   league=J1 100 Year Vision League | level=— | country=— | value=€21.30m | status

In [1]:
"""
SCRAPE LEAGUE INFO FROM TRANSFERMARKT
Reads league_url from your clubs CSV, visits each UNIQUE league page once,
extracts country and tier, then merges back into the clubs file.

Input:  clubs_info.csv   (must have 'league_url' column)
Output: clubs_info.csv   (same file, adds 'league_country' and 'league_level')
        leagues_cache.csv (one row per league — reused on re-runs)

Usage:
    python scrape_leagues.py
"""

import os
import re
import time
import random

import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# ── CONFIG ───────────────────────────────────────────────────────────────────
INPUT_CSV   = "leagues.csv"      # your clubs file with league_url column
OUTPUT_CSV  = "leagues_info.csv"      # overwrite in place (or change to a new name)
CACHE_CSV   = "leagues_cache.csv"   # one row per league — safe to delete to re-scrape
PAGE_WAIT   = 4
# ─────────────────────────────────────────────────────────────────────────────


# ── DRIVER ───────────────────────────────────────────────────────────────────

def make_driver():
    opts = Options()
    opts.add_argument("--headless=new")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--disable-blink-features=AutomationControlled")
    opts.add_experimental_option("excludeSwitches", ["enable-automation"])
    opts.add_experimental_option("useAutomationExtension", False)
    opts.add_argument("--window-size=1920,1080")
    opts.add_argument("--lang=en-US")
    opts.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
    )
    driver = webdriver.Chrome(options=opts)
    driver.execute_script(
        "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
    )
    return driver


def get_soup(driver, url: str) -> BeautifulSoup:
    driver.get(url)
    try:
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "div.data-header"))
        )
    except Exception:
        pass
    time.sleep(PAGE_WAIT)
    return BeautifulSoup(driver.page_source, "lxml")


# ── LEAGUE PAGE EXTRACTOR ─────────────────────────────────────────────────────

def scrape_league_page(driver, league_url: str) -> dict:
    """
    Visit a Transfermarkt league page and extract country + tier.

    The page text looks like:
        LaLiga
        Spain          ← country (as a link to /wettbewerbe/national/...)
        League level: First Tier
        Reigning champion: FC Barcelona
        ...
    """
    info = {
        "league_url":     league_url,
        "league_country": "",
        "league_level":   "",
        "league_name_tm": "",   # official name from Transfermarkt
    }

    try:
        soup = get_soup(driver, league_url)
        text = soup.get_text(separator="\n")

        # ── League name from <title> ──────────────────────────────────────
        title = soup.find("title")
        if title:
            m = re.match(r"^(.+?)\s*[-–|]", title.get_text())
            if m:
                info["league_name_tm"] = m.group(1).strip()

        # ── Country ───────────────────────────────────────────────────────
        # Strategy 1: exact href  /wettbewerbe/national/wettbewerbe/NNN
        country_tag = soup.find("a", href=re.compile(r"/wettbewerbe/national/wettbewerbe/\d+"))
        if country_tag:
            info["league_country"] = country_tag.get_text(strip=True)

        # Strategy 2: flag image with title attribute  <img title="Spain" ...>
        if not info["league_country"]:
            flag = soup.find("img", {"class": re.compile(r"flagge|flag", re.I)})
            if flag and flag.get("title"):
                info["league_country"] = flag["title"].strip()

        # Strategy 3: label/value pair in page text
        if not info["league_country"]:
            for label in soup.find_all(string=re.compile(r"^Country:?$", re.I)):
                parent = label.find_parent()
                if parent:
                    nxt = parent.find_next_sibling()
                    if nxt:
                        candidate = nxt.get_text(strip=True)
                        if candidate and len(candidate) < 50:
                            info["league_country"] = candidate
                            break

        # Strategy 4: regex on page text
        if not info["league_country"]:
            m = re.search(r"Country:\s*\n?\s*([A-Za-z\u00C0-\u017E ]+)", text)
            if m:
                candidate = m.group(1).strip()
                if len(candidate) < 50:
                    info["league_country"] = candidate

        # Strategy 5: first /wettbewerbe/ link after h1 that looks like a country name
        if not info["league_country"]:
            h1 = soup.find("h1")
            if h1:
                for a in h1.find_all_next("a", href=re.compile(r"/wettbewerbe/")):
                    txt = a.get_text(strip=True)
                    if txt and not txt.isdigit() and "Tier" not in txt and len(txt) < 40:
                        info["league_country"] = txt
                        break

        # Strategy 6: debug dump — print all /wettbewerbe/ links so we can see what's there
        if not info["league_country"]:
            wettbewerbe_links = soup.find_all("a", href=re.compile(r"/wettbewerbe/"))
            if wettbewerbe_links:
                print(f"   ℹ️  /wettbewerbe/ links found: " +
                      " | ".join(f"'{a.get_text(strip=True)}' ({a.get('href','')})"
                                 for a in wettbewerbe_links[:5]))

        # ── League level / tier ───────────────────────────────────────────
        # Page text: "League level: First Tier"
        m = re.search(
            r"League level:\s*\n?\s*((First|Second|Third|Fourth|Fifth|Sixth|Seventh)\s+Tier)",
            text, re.I
        )
        if m:
            info["league_level"] = m.group(1).strip()

        # Also try the link approach (same as club page)
        if not info["league_level"]:
            for a in soup.find_all("a", href=re.compile(r"/wettbewerb/")):
                txt = a.get_text(strip=True)
                if re.match(r"(First|Second|Third|Fourth|Fifth|Sixth|Seventh)\s+Tier", txt, re.I):
                    info["league_level"] = txt
                    break

        print(
            f"   ✅ {info['league_name_tm']:35s} "
            f"country={info['league_country'] or '?':20s} "
            f"tier={info['league_level'] or '?'}"
        )

    except Exception as e:
        print(f"   ❌ Error: {str(e)[:100]}")
        info["league_level"] = "error"

    return info


# ── MAIN ─────────────────────────────────────────────────────────────────────

def run():
    print("=" * 65)
    print("  LEAGUE COUNTRY & TIER SCRAPER")
    print("=" * 65)

    # ── Load clubs CSV ────────────────────────────────────────────────────
    clubs_df = None
    for enc in ["utf-8", "latin-1", "cp1252"]:
        try:
            clubs_df = pd.read_csv(INPUT_CSV, encoding=enc)
            break
        except Exception:
            continue
    if clubs_df is None:
        print(f"ERROR: could not load {INPUT_CSV}")
        return

    if "league_url" not in clubs_df.columns:
        print("ERROR: 'league_url' column not found in CSV")
        print(f"  Available columns: {list(clubs_df.columns)}")
        return

    # ── Find unique league URLs to scrape ─────────────────────────────────
    all_urls = (
        clubs_df["league_url"]
        .dropna()
        .astype(str)
        .pipe(lambda s: s[s.str.startswith("https://")])
        .unique()
        .tolist()
    )
    print(f"  Clubs: {len(clubs_df)}")
    print(f"  Unique league URLs: {len(all_urls)}")

    # ── Load cache (leagues already scraped in a previous run) ────────────
    cache: dict[str, dict] = {}
    if os.path.exists(CACHE_CSV):
        cache_df = pd.read_csv(CACHE_CSV)
        for _, row in cache_df.iterrows():
            cache[str(row["league_url"])] = row.to_dict()
        print(f"  Loaded {len(cache)} leagues from cache ({CACHE_CSV})")

    to_scrape = [u for u in all_urls if u not in cache]
    print(f"  To scrape: {len(to_scrape)}")

    # ── Scrape missing leagues ─────────────────────────────────────────────
    if to_scrape:
        driver = make_driver()
        try:
            for i, url in enumerate(to_scrape, 1):
                print(f"\n[{i}/{len(to_scrape)}] {url}")
                info = scrape_league_page(driver, url)
                cache[url] = info

                # Save cache after every league (it's a small file)
                pd.DataFrame(list(cache.values())).to_csv(CACHE_CSV, index=False)

                if i < len(to_scrape):
                    time.sleep(random.uniform(3, 6))
        except Exception as e:
            print(f"\n💥 Unexpected error: {e}")
        finally:
            try:
                driver.quit()
            except Exception:
                pass
            print("\n✅ Driver closed.")
    else:
        print("  Nothing new to scrape — all leagues already in cache.")

    # ── Merge back into clubs DataFrame ───────────────────────────────────
    print("\n📊 Merging into clubs CSV …")

    cache_df = pd.DataFrame(list(cache.values()))

    # Only bring in the two columns we want (don't clobber existing ones)
    merge_cols = ["league_url", "league_country", "league_level"]
    available  = [c for c in merge_cols if c in cache_df.columns]

    # Drop the old columns so the merge replaces them cleanly
    for col in ["league_country", "league_level"]:
        if col in clubs_df.columns:
            clubs_df = clubs_df.drop(columns=[col])

    merged = clubs_df.merge(
        cache_df[available].drop_duplicates("league_url"),
        on="league_url",
        how="left"
    )

    merged.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

    # ── Summary ───────────────────────────────────────────────────────────
    filled_country = merged["league_country"].notna() & (merged["league_country"] != "")
    filled_tier    = merged["league_level"].notna()   & (merged["league_level"]   != "")

    print(f"\n  Clubs with country: {filled_country.sum()} / {len(merged)}")
    print(f"  Clubs with tier:    {filled_tier.sum()} / {len(merged)}")
    print(f"\n💾 Saved → {OUTPUT_CSV}")
    print(f"💾 League cache → {CACHE_CSV}")

    # Preview
    preview_cols = ["club_name" if "club_name" in merged.columns else merged.columns[0],
                    "league", "league_country", "league_level"]
    preview_cols = [c for c in preview_cols if c in merged.columns]
    print(f"\n{'Sample output':}")
    print(merged[preview_cols].head(10).to_string(index=False))


if __name__ == "__main__":
    run()

  LEAGUE COUNTRY & TIER SCRAPER
  Clubs: 210
  Unique league URLs: 209
  Loaded 44 leagues from cache (leagues_cache.csv)
  To scrape: 165

[1/165] https://www.transfermarkt.com/allsvenskan/startseite/wettbewerb/SE1
   ✅ Allsvenskan 2026                    country=Sweden               tier=First Tier

[2/165] https://www.transfermarkt.com/nasl-spring-championship/startseite/wettbewerb/NASC
   ✅ NASL Spring Championship (2013      country=United States        tier=Second Tier

[3/165] https://www.transfermarkt.com/liga-mx-apertura/startseite/wettbewerb/MEXA
   ✅ Liga MX Apertura 25/26              country=Mexico               tier=First Tier

[4/165] https://www.transfermarkt.com/primera-division-de-chile/startseite/wettbewerb/CLPD
   ✅ Liga de Primera  2026               country=Chile                tier=First Tier

[5/165] https://www.transfermarkt.com/primera-division-apertura/startseite/wettbewerb/PR1A
   ✅ Primera División Apertura 2026      country=Paraguay             tier=First 